# OpenVoice Training for Twi Voice Synthesis
## High-Quality Multilingual Voice Cloning

This notebook implements OpenVoice training for Twi using processed 24kHz audio data.
OpenVoice features:
- Instant voice cloning with short audio reference
- Multilingual support with accent transfer
- High-quality 24kHz audio synthesis
- Flexible voice style control

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create and set the working directory
BASE_PATH = '/content/drive/MyDrive/voice-lab-data/'
os.makedirs(BASE_PATH, exist_ok=True)
os.chdir(BASE_PATH)
print(f"Working directory set to: {os.getcwd()}")

In [ ]:
%%writefile openvoice_trainer.py

import json
import time
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

@dataclass
class OpenVoiceConfig:
    """Configuration for OpenVoice training."""
    base_path: Path
    processed_data_path: Path
    model_path: Path
    output_path: Path
    
    # Audio configuration optimized for OpenVoice
    sample_rate: int = 24000  # High quality for OpenVoice
    hop_length: int = 240     # 24000/100 for 10ms frames
    win_length: int = 960     # 40ms window
    n_fft: int = 1024
    n_mels: int = 100
    
    # Training parameters
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    batch_size: int = 16 if torch.cuda.is_available() else 4
    learning_rate: float = 1e-4
    warmup_steps: int = 4000
    max_steps: int = 200000
    save_interval: int = 5000
    log_interval: int = 100
    
    # Quality filters
    min_duration: float = 1.0
    max_duration: float = 15.0
    min_text_length: int = 10
    max_text_length: int = 300
    
    def __post_init__(self):
        self.output_path.mkdir(parents=True, exist_ok=True)
        self.model_path.mkdir(parents=True, exist_ok=True)
        
        # Adjust batch size for GPU memory
        if torch.cuda.is_available():
            gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
            if gpu_memory_gb < 8: 
                self.batch_size = 4
            elif gpu_memory_gb < 16: 
                self.batch_size = 8
            else: 
                self.batch_size = 16

class OpenVoiceDataLoader:
    """Data loader for OpenVoice training using processed 24kHz data."""
    
    def __init__(self, config: OpenVoiceConfig):
        self.config = config
        
    def load_processed_dataset(self) -> pd.DataFrame:
        """Load 24kHz processed dataset optimized for OpenVoice."""
        print("🔄 Loading processed dataset for OpenVoice (24kHz)...")
        
        # Try to load from processed 24kHz data
        processed_manifest = self.config.processed_data_path / 'manifests' / 'tts_24k_dataset.csv'
        
        if processed_manifest.exists():
            print(f"   ✅ Loading from processed 24kHz manifest: {processed_manifest}")
            df = pd.read_csv(processed_manifest)
            
            # Validate required columns
            required_columns = ['processed_audio_path', 'processed_text', 'speaker_id', 'processed_duration']
            if all(col in df.columns for col in required_columns):
                # Apply quality filters
                df_filtered = self._apply_quality_filters(df)
                print(f"   ✅ Loaded {len(df_filtered)} high-quality samples for OpenVoice")
                return df_filtered
            else:
                print(f"   ⚠️ Missing required columns in processed data")
        
        # Try OpenVoice-specific manifests
        openvoice_train = self.config.processed_data_path / 'model_manifests' / 'openvoice_train.csv'
        openvoice_val = self.config.processed_data_path / 'model_manifests' / 'openvoice_val.csv'
        
        if openvoice_train.exists() and openvoice_val.exists():
            print(f"   ✅ Loading from OpenVoice-specific manifests")
            train_df = pd.read_csv(openvoice_train)
            val_df = pd.read_csv(openvoice_val)
            train_df['split'] = 'train'
            val_df['split'] = 'val'
            df = pd.concat([train_df, val_df], ignore_index=True)
            
            if not df.empty:
                df_filtered = self._apply_quality_filters(df)
                print(f"   ✅ Loaded {len(df_filtered)} samples from OpenVoice manifests")
                return df_filtered
        
        # Final fallback: create dummy dataset
        print("   ⚠️ No processed 24kHz data found, creating dummy dataset")
        return self._create_dummy_dataset()
    
    def _apply_quality_filters(self, df: pd.DataFrame) -> pd.DataFrame:
        """Apply quality filters optimized for OpenVoice."""
        print("   🔍 Applying OpenVoice quality filters...")
        
        initial_count = len(df)
        
        # Ensure we have the necessary columns
        if 'processed_duration' not in df.columns:
            df['processed_duration'] = 3.0  # Default duration
        if 'text_length' not in df.columns:
            df['text_length'] = df['processed_text'].str.len()
        
        # Apply filters
        df_filtered = df[
            (df['processed_duration'] >= self.config.min_duration) &
            (df['processed_duration'] <= self.config.max_duration) &
            (df['text_length'] >= self.config.min_text_length) &
            (df['text_length'] <= self.config.max_text_length)
        ].copy()
        
        retained_count = len(df_filtered)
        retention_rate = (retained_count / initial_count * 100) if initial_count > 0 else 0
        
        print(f"   Filtered: {initial_count} → {retained_count} samples ({retention_rate:.1f}% retained)")
        return df_filtered
    
    def _create_dummy_dataset(self) -> pd.DataFrame:
        """Create dummy dataset for OpenVoice testing."""
        print("   📝 Creating dummy 24kHz dataset for OpenVoice...")
        
        dummy_dir = self.config.output_path / 'dummy_openvoice'
        dummy_dir.mkdir(parents=True, exist_ok=True)
        
        # High-quality Twi samples for OpenVoice
        samples = [
            ("Wo ho te sɛn? Me ho yɛ paa.", "speaker_001"),
            ("Me ma wo akye. Nante yie na yɛbɛhunu wo bio.", "speaker_002"),
            ("Twi kasa yɛ dɛ yiye. Me pɛ sɛ meka Twi berɛ biara.", "speaker_003"),
            ("Me da wo ase yiye. Wo ayɛ adwuma pa ara.", "speaker_004"),
            ("Yɛn ara yɛ Ghanafoɔ. Yɛn kasa ne Twi.", "speaker_005"),
            ("Adwo paa! Sɛn na wo te? Me nso me ho yɛ.", "speaker_006"),
            ("Meda wo ase sɛ woba bɛsraa me nnɛ.", "speaker_007"),
            ("Ghana yɛ ɔman fɛfɛ. Me dɔ me man no.", "speaker_008")
        ]
        
        dummy_data = []
        for i, (text, speaker) in enumerate(samples):
            audio_file = dummy_dir / f"{speaker}_{i:03d}.wav"
            
            if not audio_file.exists():
                # Generate high-quality dummy audio at 24kHz
                duration = max(3.0, len(text) * 0.08)  # Realistic duration
                dummy_audio = self._generate_realistic_audio(duration, self.config.sample_rate)
                sf.write(audio_file, dummy_audio, self.config.sample_rate)
            
            info = sf.info(str(audio_file))
            dummy_data.append({
                'processed_audio_path': str(audio_file),
                'processed_text': text,
                'speaker_id': speaker,
                'processed_duration': info.duration,
                'processed_sample_rate': info.samplerate,
                'text_length': len(text),
                'language': 'tw',
                'split': 'train' if i < 6 else 'val'
            })
        
        return pd.DataFrame(dummy_data)
    
    def _generate_realistic_audio(self, duration: float, sr: int) -> np.ndarray:
        """Generate more realistic dummy audio with speech-like characteristics."""
        samples = int(duration * sr)
        
        # Generate base signal with multiple frequencies
        t = np.linspace(0, duration, samples, False)
        
        # Fundamental frequency around human speech
        f0 = 150 + 50 * np.sin(2 * np.pi * 2 * t)  # Varying F0
        signal = 0.3 * np.sin(2 * np.pi * f0 * t)
        
        # Add harmonics
        signal += 0.2 * np.sin(2 * np.pi * 2 * f0 * t)
        signal += 0.1 * np.sin(2 * np.pi * 3 * f0 * t)
        
        # Add noise for realism
        noise = np.random.normal(0, 0.02, samples)
        signal += noise
        
        # Apply envelope for speech-like amplitude modulation
        envelope = 0.5 + 0.5 * np.cos(2 * np.pi * 5 * t + np.pi)
        envelope = np.clip(envelope, 0.1, 1.0)
        signal *= envelope
        
        # Normalize
        signal = signal / np.max(np.abs(signal)) * 0.7
        
        return signal.astype(np.float32)

class OpenVoiceTrainer:
    """OpenVoice model trainer for Twi."""
    
    def __init__(self, config: OpenVoiceConfig, dataset: pd.DataFrame):
        self.config = config
        self.dataset = dataset
        self.model = None
    
    def prepare_training_data(self) -> Dict[str, Path]:
        """Prepare training data in OpenVoice format."""
        print("\n📄 Preparing OpenVoice training data...")
        
        # Create training directory structure
        train_dir = self.config.output_path / 'openvoice_training_data'
        audio_dir = train_dir / 'audio'
        text_dir = train_dir / 'text'
        
        audio_dir.mkdir(parents=True, exist_ok=True)
        text_dir.mkdir(parents=True, exist_ok=True)
        
        # Process dataset
        train_samples = []
        val_samples = []
        
        for _, row in tqdm(self.dataset.iterrows(), total=len(self.dataset), desc="Processing samples"):
            try:
                src_audio = Path(row['processed_audio_path'])
                if not src_audio.exists():
                    continue
                
                # Create unique filename
                filename = f"{row['speaker_id']}_{src_audio.stem}"
                
                # Copy audio file
                dst_audio = audio_dir / f"{filename}.wav"
                if not dst_audio.exists():
                    # Ensure correct sample rate
                    audio, sr = librosa.load(str(src_audio), sr=self.config.sample_rate)
                    sf.write(dst_audio, audio, self.config.sample_rate)
                
                # Create text file
                text_file = text_dir / f"{filename}.txt"
                text_file.write_text(row['processed_text'], encoding='utf-8')
                
                # Create sample record
                sample_record = {
                    'filename': filename,
                    'audio_path': str(dst_audio),
                    'text_path': str(text_file),
                    'text': row['processed_text'],
                    'speaker_id': row['speaker_id'],
                    'duration': row['processed_duration']
                }
                
                # Split into train/val
                if row.get('split', 'train') == 'train':
                    train_samples.append(sample_record)
                else:
                    val_samples.append(sample_record)
                    
            except Exception as e:
                print(f"   ⚠️ Error processing {row.get('processed_audio_path', 'unknown')}: {e}")
                continue
        
        # Save manifests
        train_manifest = train_dir / 'train_manifest.json'
        val_manifest = train_dir / 'val_manifest.json'
        
        with open(train_manifest, 'w', encoding='utf-8') as f:
            json.dump(train_samples, f, indent=2, ensure_ascii=False)
        
        with open(val_manifest, 'w', encoding='utf-8') as f:
            json.dump(val_samples, f, indent=2, ensure_ascii=False)
        
        print(f"   ✅ Training data prepared:")
        print(f"      Train samples: {len(train_samples)}")
        print(f"      Val samples: {len(val_samples)}")
        print(f"      Audio directory: {audio_dir}")
        print(f"      Text directory: {text_dir}")
        
        return {
            'train_dir': train_dir,
            'audio_dir': audio_dir,
            'text_dir': text_dir,
            'train_manifest': train_manifest,
            'val_manifest': val_manifest
        }
    
    def generate_training_config(self, data_paths: Dict[str, Path]) -> Path:
        """Generate OpenVoice training configuration."""
        print("\n⚙️ Generating OpenVoice training configuration...")
        
        config = {
            "model": {
                "name": "OpenVoice",
                "version": "v2",
                "language": "twi"
            },
            "audio": {
                "sample_rate": self.config.sample_rate,
                "hop_length": self.config.hop_length,
                "win_length": self.config.win_length,
                "n_fft": self.config.n_fft,
                "n_mels": self.config.n_mels,
                "mel_fmin": 0,
                "mel_fmax": self.config.sample_rate // 2
            },
            "training": {
                "batch_size": self.config.batch_size,
                "learning_rate": self.config.learning_rate,
                "warmup_steps": self.config.warmup_steps,
                "max_steps": self.config.max_steps,
                "save_interval": self.config.save_interval,
                "log_interval": self.config.log_interval,
                "gradient_clip": 1.0,
                "mixed_precision": True
            },
            "data": {
                "train_manifest": str(data_paths['train_manifest']),
                "val_manifest": str(data_paths['val_manifest']),
                "audio_dir": str(data_paths['audio_dir']),
                "text_dir": str(data_paths['text_dir'])
            },
            "model_config": {
                "hidden_dim": 256,
                "encoder_layers": 6,
                "decoder_layers": 6,
                "attention_heads": 8,
                "dropout": 0.1
            },
            "paths": {
                "checkpoint_dir": str(self.config.model_path),
                "log_dir": str(self.config.output_path / 'logs')
            }
        }
        
        # Save configuration
        config_path = self.config.output_path / 'openvoice_config.json'
        with open(config_path, 'w', encoding='utf-8') as f:
            json.dump(config, f, indent=2, ensure_ascii=False)
        
        print(f"   ✅ Configuration saved to: {config_path}")
        return config_path
    
    def generate_training_script(self, config_path: Path) -> Path:
        """Generate training script for OpenVoice."""
        print("\n📜 Generating OpenVoice training script...")
        
        script_content = f"""#!/bin/bash
# OpenVoice Training Script for Twi
# Generated for high-quality 24kHz voice synthesis

echo "Starting OpenVoice training for Twi..."

# Set environment
export CUDA_VISIBLE_DEVICES=0
export PYTHONPATH=$PYTHONPATH:/content

# Training configuration
CONFIG_PATH="{config_path}"
CHECKPOINT_DIR="{self.config.model_path}"
LOG_DIR="{self.config.output_path}/logs"

# Create directories
mkdir -p "$CHECKPOINT_DIR"
mkdir -p "$LOG_DIR"

echo "Configuration: $CONFIG_PATH"
echo "Checkpoints: $CHECKPOINT_DIR"
echo "Logs: $LOG_DIR"

# Note: This is a template script. 
# Install OpenVoice from: https://github.com/myshell-ai/OpenVoice
# Then modify this script to use the actual OpenVoice training command

# Example training command (adjust based on OpenVoice implementation):
# python -m openvoice.train \\
#     --config "$CONFIG_PATH" \\
#     --checkpoint_dir "$CHECKPOINT_DIR" \\
#     --log_dir "$LOG_DIR" \\
#     --gpu 0

echo "Training preparation complete!"
echo "Please install OpenVoice and run the training command above."
"""
        
        script_path = self.config.output_path / 'train_openvoice.sh'
        script_path.write_text(script_content)
        script_path.chmod(0o755)  # Make executable
        
        print(f"   ✅ Training script saved to: {script_path}")
        return script_path

class OpenVoiceInference:
    """OpenVoice inference engine for voice cloning."""
    
    def __init__(self, config: OpenVoiceConfig):
        self.config = config
        self.model = None
        
    def load_model(self, checkpoint_path: Optional[str] = None):
        """Load OpenVoice model for inference."""
        print("\n🤖 Loading OpenVoice model...")
        
        # Since OpenVoice requires specific installation,
        # we'll create a mock model for demonstration
        self.model = self._create_mock_model()
        print("   ⚠️ Using mock OpenVoice model for demonstration")
        print("   📝 To use real OpenVoice, install from: https://github.com/myshell-ai/OpenVoice")
    
    def _create_mock_model(self):
        """Create mock OpenVoice model for testing."""
        class MockOpenVoice:
            def __init__(self, config):
                self.config = config
            
            def clone_voice(self, text: str, reference_audio: str, output_path: str, language: str = "tw"):
                """Mock voice cloning."""
                # Generate high-quality dummy audio
                duration = max(3.0, len(text) * 0.08)
                samples = int(duration * self.config.sample_rate)
                
                # Create more realistic audio with speech patterns
                t = np.linspace(0, duration, samples, False)
                
                # Base speech-like signal
                f0 = 140 + 30 * np.sin(2 * np.pi * 1.5 * t)
                signal = 0.4 * np.sin(2 * np.pi * f0 * t)
                signal += 0.2 * np.sin(2 * np.pi * 2 * f0 * t)
                signal += 0.1 * np.sin(2 * np.pi * 3 * f0 * t)
                
                # Add formants (simplified)
                formant1 = 0.1 * np.sin(2 * np.pi * 800 * t)
                formant2 = 0.05 * np.sin(2 * np.pi * 1200 * t)
                signal += formant1 + formant2
                
                # Speech envelope
                envelope = 0.6 + 0.4 * np.random.random(samples)
                envelope = np.convolve(envelope, np.ones(100)/100, mode='same')  # Smooth
                signal *= envelope
                
                # Add realistic noise
                noise = np.random.normal(0, 0.01, samples)
                signal += noise
                
                # Normalize
                signal = signal / np.max(np.abs(signal)) * 0.8
                
                # Save
                sf.write(output_path, signal.astype(np.float32), self.config.sample_rate)
                return output_path
        
        return MockOpenVoice(self.config)
    
    def test_voice_cloning(self, dataset: pd.DataFrame, num_tests: int = 3) -> List[Dict]:
        """Test OpenVoice voice cloning capabilities."""
        print(f"\n🧪 Testing OpenVoice cloning with {num_tests} examples...")
        
        if dataset.empty:
            print("   ⚠️ No dataset available for testing")
            return []
        
        # Test phrases in Twi
        test_phrases = [
            "Wo ho te sɛn? Me ho yɛ paa, me da wo ase.",
            "Ghana yɛ ɔman fɛfɛ. Me dɔ me man no yiye.",
            "Twi kasa yɛ dɛ. Yɛbɛtoa so aka Twi.",
            "Nante yie na yɛbɛhunu wo bio daakye.",
            "Me ma wo akye. Ɛyɛ sɛn na wo nna te?"
        ]
        
        # Get reference audios
        available_refs = dataset[dataset['processed_audio_path'].apply(lambda x: Path(x).exists())]
        if len(available_refs) == 0:
            print("   ⚠️ No reference audio files available")
            return []
        
        results = []
        
        for i in range(min(num_tests, len(test_phrases), len(available_refs))):
            text = test_phrases[i]
            ref_audio = available_refs.iloc[i]['processed_audio_path']
            speaker_id = available_refs.iloc[i]['speaker_id']
            
            output_path = self.config.output_path / f"openvoice_clone_{i+1}_{int(time.time())}.wav"
            
            print(f"   🎤 Test {i+1}: Cloning with {speaker_id}")
            print(f"      Text: '{text[:50]}...'")
            
            start_time = time.time()
            
            try:
                self.model.clone_voice(
                    text=text,
                    reference_audio=ref_audio,
                    output_path=str(output_path),
                    language="tw"
                )
                
                inference_time = time.time() - start_time
                audio_info = sf.info(str(output_path))
                
                result = {
                    'test_id': i + 1,
                    'text': text,
                    'reference_speaker': speaker_id,
                    'reference_audio': str(ref_audio),
                    'output_path': str(output_path),
                    'inference_time': inference_time,
                    'audio_duration': audio_info.duration,
                    'audio_sample_rate': audio_info.samplerate,
                    'rtf': inference_time / audio_info.duration if audio_info.duration > 0 else 0,
                    'success': True
                }
                
                results.append(result)
                
                print(f"      ✅ Generated in {inference_time:.2f}s")
                print(f"      📊 Audio: {audio_info.duration:.2f}s @ {audio_info.samplerate}Hz")
                print(f"      ⚡ RTF: {result['rtf']:.2f}x")
                
            except Exception as e:
                print(f"      ❌ Error: {e}")
                results.append({
                    'test_id': i + 1,
                    'text': text,
                    'reference_speaker': speaker_id,
                    'error': str(e),
                    'success': False
                })
        
        # Save results
        results_path = self.config.output_path / 'openvoice_test_results.json'
        with open(results_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False, default=str)
        
        print(f"\n📄 Test results saved to: {results_path}")
        return results

# Utility functions
def setup_openvoice_training(trainer: OpenVoiceTrainer) -> Dict[str, Path]:
    """Complete OpenVoice training setup."""
    print("\n🚀 Setting up OpenVoice training pipeline...")
    
    try:
        # Prepare training data
        data_paths = trainer.prepare_training_data()
        
        # Generate configuration
        config_path = trainer.generate_training_config(data_paths)
        data_paths['config'] = config_path
        
        # Generate training script
        script_path = trainer.generate_training_script(config_path)
        data_paths['script'] = script_path
        
        print("\n✅ OpenVoice training setup complete!")
        return data_paths
        
    except Exception as e:
        print(f"\n❌ Error setting up training: {e}")
        import traceback
        traceback.print_exc()
        return {}

print("✅ OpenVoice training utilities loaded successfully!")

In [ ]:
# Install required packages
print("📦 Installing packages for OpenVoice...")
!pip install -q torch torchaudio transformers
!pip install -q soundfile librosa pandas numpy tqdm
!pip install -q datasets pyarrow

print("\n✅ Packages installed successfully!")
print("📝 Note: OpenVoice requires specific installation from GitHub")
print("   Repository: https://github.com/myshell-ai/OpenVoice")

In [ ]:
# Import OpenVoice utilities
from openvoice_trainer import (
    OpenVoiceConfig,
    OpenVoiceDataLoader,
    OpenVoiceTrainer,
    OpenVoiceInference,
    setup_openvoice_training
)

import torch
import pandas as pd
from pathlib import Path

print("✅ OpenVoice utilities imported successfully!")
print(f"   PyTorch Version: {torch.__version__}")
print(f"   CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Initialize OpenVoice configuration
BASE_PATH = Path('/content/drive/MyDrive/voice-lab-data')

openvoice_config = OpenVoiceConfig(
    base_path=BASE_PATH,
    processed_data_path=BASE_PATH / 'processed',
    model_path=BASE_PATH / 'models' / 'openvoice',
    output_path=BASE_PATH / 'output' / 'openvoice'
)

print(f"🔧 OpenVoice Configuration:")
print(f"   Sample rate: {openvoice_config.sample_rate}Hz (high quality)")
print(f"   Batch size: {openvoice_config.batch_size}")
print(f"   Learning rate: {openvoice_config.learning_rate}")
print(f"   Max steps: {openvoice_config.max_steps:,}")
print(f"   Device: {openvoice_config.device}")
print(f"   Model path: {openvoice_config.model_path}")
print(f"   Output path: {openvoice_config.output_path}")

In [ ]:
# Load processed 24kHz dataset
print("📊 Loading processed 24kHz dataset for OpenVoice...")

data_loader = OpenVoiceDataLoader(openvoice_config)
openvoice_dataset = data_loader.load_processed_dataset()

print(f"\n📈 OpenVoice dataset loaded:")
print(f"   Total samples: {len(openvoice_dataset):,}")

if not openvoice_dataset.empty:
    print(f"   Unique speakers: {openvoice_dataset['speaker_id'].nunique():,}")
    
    if 'processed_duration' in openvoice_dataset.columns:
        total_duration = openvoice_dataset['processed_duration'].sum()
        print(f"   Total duration: {total_duration/3600:.1f} hours")
        print(f"   Average duration: {openvoice_dataset['processed_duration'].mean():.1f}s")
    
    if 'processed_sample_rate' in openvoice_dataset.columns:
        sample_rates = openvoice_dataset['processed_sample_rate'].unique()
        print(f"   Sample rates: {sample_rates}")
    
    # Show train/val split
    if 'split' in openvoice_dataset.columns:
        split_counts = openvoice_dataset['split'].value_counts()
        print(f"   Train/Val split: {split_counts.to_dict()}")
    
    # Display sample
    print(f"\n📝 Sample records:")
    display(openvoice_dataset.head())
else:
    print("   ⚠️ Using dummy 24kHz dataset for testing")

In [ ]:
# Initialize OpenVoice trainer
print("🎓 Initializing OpenVoice trainer...")

openvoice_trainer = OpenVoiceTrainer(openvoice_config, openvoice_dataset)
print("   ✅ OpenVoice trainer initialized!")

# Setup complete training pipeline
print("\n🚀 Setting up OpenVoice training pipeline...")
training_paths = setup_openvoice_training(openvoice_trainer)

if training_paths:
    print("\n✅ OpenVoice training pipeline ready!")
    print(f"   📁 Training directory: {training_paths.get('train_dir', 'N/A')}")
    print(f"   🎵 Audio directory: {training_paths.get('audio_dir', 'N/A')}")
    print(f"   📝 Text directory: {training_paths.get('text_dir', 'N/A')}")
    print(f"   ⚙️ Config file: {training_paths.get('config', 'N/A')}")
    print(f"   📜 Training script: {training_paths.get('script', 'N/A')}")
else:
    print("\n⚠️ Training pipeline setup incomplete")

In [ ]:
# Initialize and test OpenVoice inference
print("🤖 Testing OpenVoice inference capabilities...")

openvoice_inference = OpenVoiceInference(openvoice_config)
openvoice_inference.load_model()

# Test voice cloning
if not openvoice_dataset.empty:
    print("\n🧪 Running OpenVoice cloning tests...")
    cloning_results = openvoice_inference.test_voice_cloning(openvoice_dataset, num_tests=3)
    
    if cloning_results:
        print(f"\n📊 Cloning test summary:")
        successful_tests = [r for r in cloning_results if r.get('success', False)]
        print(f"   Successful tests: {len(successful_tests)}/{len(cloning_results)}")
        
        if successful_tests:
            avg_rtf = sum(r['rtf'] for r in successful_tests) / len(successful_tests)
            avg_duration = sum(r['audio_duration'] for r in successful_tests) / len(successful_tests)
            print(f"   Average RTF: {avg_rtf:.2f}x")
            print(f"   Average audio duration: {avg_duration:.1f}s")
            print(f"   Sample rate: {successful_tests[0]['audio_sample_rate']}Hz")
    else:
        print("   ⚠️ No cloning tests completed")
else:
    print("\n⚠️ No dataset available for cloning tests")

In [ ]:
# Display training instructions and next steps
print("\n📋 OpenVoice Training Instructions")
print("=" * 50)

print(f"\n🔧 Setup Complete:")
if not openvoice_dataset.empty:
    print(f"   ✅ Dataset: {len(openvoice_dataset):,} samples at 24kHz")
    print(f"   ✅ Speakers: {openvoice_dataset['speaker_id'].nunique():,} unique voices")
    if 'processed_duration' in openvoice_dataset.columns:
        total_hours = openvoice_dataset['processed_duration'].sum() / 3600
        print(f"   ✅ Duration: {total_hours:.1f} hours of audio")
else:
    print(f"   ⚠️ Using dummy dataset for demonstration")

if training_paths:
    print(f"   ✅ Training data prepared and structured")
    print(f"   ✅ Configuration files generated")
    print(f"   ✅ Training script created")

print(f"\n📝 To Train OpenVoice:")
print(f"   1. Install OpenVoice:")
print(f"      git clone https://github.com/myshell-ai/OpenVoice.git")
print(f"      cd OpenVoice")
print(f"      pip install -e .")
print(f"")
print(f"   2. Run training:")
if training_paths and 'script' in training_paths:
    print(f"      chmod +x {training_paths['script']}")
    print(f"      {training_paths['script']}")
else:
    print(f"      # Use generated configuration and data")
print(f"")
print(f"   3. Monitor training:")
print(f"      # Check logs in: {openvoice_config.output_path}/logs")
print(f"      # Checkpoints in: {openvoice_config.model_path}")

print(f"\n🎯 OpenVoice Features for Twi:")
print(f"   • High-quality 24kHz audio synthesis")
print(f"   • Instant voice cloning with short references")
print(f"   • Multilingual support with accent control")
print(f"   • Style and emotion transfer capabilities")
print(f"   • Real-time inference performance")

print(f"\n✨ Ready for OpenVoice training with Twi data!")